In [2]:
import os
import random
import math
from datetime import datetime
from collections import Counter
import pandas as pd
import numpy as np

import cv2
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

In [3]:
import pandas as pd
import os
dossier = 'train/labels'
colnames=['Coin h-g x', 'Coin h-g y', 'Coin b-d x','Coin b-d y','classe','nom']

dataframes = []
for fichier in os.listdir(dossier):
    if fichier.endswith('.csv'):
        chemin_fichier = os.path.join(dossier, fichier)
        try:
            df1 = pd.read_csv(chemin_fichier, names=colnames, header=None)
            chemin_fichier = chemin_fichier.replace("labels", "images")
            chemin_fichier = chemin_fichier.replace("csv", "jpg")
            df1['nom'] = chemin_fichier
            if not df1.empty:
                dataframes.append(df1)
        except pd.errors.EmptyDataError:
            pass

# Concaténer tous les DataFrames en un seul
if dataframes:
    df_final = pd.concat(dataframes, ignore_index=True)
    # Afficher les premières lignes du DataFrame final
print(df_final)



      Coin h-g x  Coin h-g y  Coin b-d x  Coin b-d y        classe  \
0            452         149         619         314          stop   
1            184         156         500         432    obligation   
2            395         272         469         454        frouge   
3            422         613         451         691        frouge   
4            397         349         544         474         ceder   
...          ...         ...         ...         ...           ...   
1129         421         607         477         667  interdiction   
1130         388         512         420         544         ceder   
1131         350         407         431         531         ceder   
1132         498         215         659         371    obligation   
1133         418         403         523         504    obligation   

                        nom  
0     train/images/0495.jpg  
1     train/images/0481.jpg  
2     train/images/0318.jpg  
3     train/images/0318.jpg  
4     tra

In [14]:
df_train = df_final 
df_train['classe'].value_counts()

classe
interdiction    365
danger          162
ceder           133
fvert           109
obligation      108
stop            100
frouge           90
forange          67
Name: count, dtype: int64

In [5]:
import numpy as np
def read_image(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)


def create_mask(bb, x):
    rows,cols,*_ = x.shape
    Y = np.zeros((rows, cols))
    Y[bb[0]:bb[2], bb[1]:bb[3]] = 1.
    return Y


def mask_to_bb(Y):
    """Конвертируем маску Y в bounding box'a, принимая 0 как фоновый ненулевой объект """
    cols, rows = np.nonzero(Y)
    if len(cols) == 0: 
        return np.zeros(4, dtype=np.float32)
    top_row = np.min(rows)
    left_col = np.min(cols)
    bottom_row = np.max(rows)
    right_col = np.max(cols)
    return np.array([left_col, top_row, right_col, bottom_row], dtype=np.float32)


def create_bb_array(x):
    """Генерируем массив bounding box'a из столбца train_df"""
    return np.array([x[0],x[1],x[2],x[3]])


def resize_image_bb(read_path, bb, sz):
    """Ресайзим изображение и его bounding box и записываем изображение в новый путь"""
    im = read_image(read_path)
    im_resized = cv2.resize(im, (sz, sz))
    Y_resized = cv2.resize(create_mask(bb, im), (sz, sz))
    #cv2.imwrite( cv2.cvtColor(im_resized, cv2.COLOR_RGB2BGR))
    return mask_to_bb(Y_resized)

In [6]:
IM_SIZE = 300

In [15]:
new_bbs = []



for index, row in df_train.iterrows():
    #print(index,row)
    new_bb = resize_image_bb(row['nom'], create_bb_array(row.values), IM_SIZE)

    new_bbs.append(new_bb)
    

df_train['new_bb'] = new_bbs

df_train.head()

,Coin h-g x,Coin h-g y,Coin b-d x,Coin b-d y,classe,nom,new_bb
0,452,149,619,314,stop,train/images/0495.jpg,"[135.0, 59.0, 185.0, 125.0]"
1,184,156,500,432,obligation,train/images/0481.jpg,"[59.0, 68.0, 161.0, 188.0]"
2,395,272,469,454,frouge,train/images/0318.jpg,"[118.0, 114.0, 140.0, 190.0]"
3,422,613,451,691,frouge,train/images/0318.jpg,"[126.0, 257.0, 134.0, 289.0]"
4,397,349,544,474,ceder,train/images/0324.jpg,"[119.0, 139.0, 162.0, 189.0]"


In [8]:
# Вырезаем кусок с изображения
def crop(im, r, c, target_r, target_c): 
    return im[r:r+target_r, c:c+target_c]

# Центральное вырезание 
def center_crop(x, r_pix=8):
    r, c,*_ = x.shape
    c_pix = round(r_pix*c/r)
    return crop(x, r_pix, c_pix, r-2*r_pix, c-2*c_pix)

def rotate_cv(im, deg, y=False, mode=cv2.BORDER_REFLECT):
    """ Поворачиваем наше изображение"""
    r,c,*_ = im.shape
    M = cv2.getRotationMatrix2D((c/2,r/2),deg,1)
    if y:
        return cv2.warpAffine(im, M, (c, r), borderMode=cv2.BORDER_CONSTANT)
    return cv2.warpAffine(im, M, (c, r), borderMode=mode, flags=cv2.WARP_FILL_OUTLIERS)

def random_cropXY(x, Y, r_pix=8):
    """ Возвращает случайное вырезание"""
    r, c,*_ = x.shape
    c_pix = round(r_pix * c/r)
    rand_r = random.uniform(0, 1)
    rand_c = random.uniform(0, 1)
    start_r = np.floor(2 * rand_r * r_pix).astype(int)
    start_c = np.floor(2 * rand_c * c_pix).astype(int)
    xx = crop(x, start_r, start_c, r - 2*r_pix, c - 2*c_pix)
    YY = crop(Y, start_r, start_c, r - 2*r_pix, c - 2*c_pix)
    return xx, YY

# Трансформируем нашу картинку 
def transformsXY(path, bb, is_transforms):
    x = cv2.imread(str(path)).astype(np.float32)
    x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB) / 255
    Y = create_mask(bb, x)
    if is_transforms:
        rdeg = (np.random.random()-.50) * 20
        x = rotate_cv(x, rdeg)
        Y = rotate_cv(Y, rdeg, y=True)
        if np.random.random() > 0.5: 
            x = np.fliplr(x).copy()
            Y = np.fliplr(Y).copy()
        x, Y = random_cropXY(x, Y)
    else:
        x, Y = center_crop(x), center_crop(Y)
    return x, mask_to_bb(Y)

def create_corner_rect(bb, color='red'):
    bb = np.array(bb, dtype=np.float32)
    return plt.Rectangle((bb[1], bb[0]), bb[3]-bb[1], bb[2]-bb[0], color=color,
                         fill=False, lw=3)

def show_corner_bb(im, bb):
    plt.imshow(im)
    plt.gca().add_patch(create_corner_rect(bb))

In [16]:
df_train = df_train.reset_index()
X = df_train[['nom', 'new_bb']]
Y = df_train['classe']
X_train, X_val, y_train, y_val = train_test_split(X, Y, test_size=0.2, random_state=42)
print(X, X_train)

def normalize(im):
    """Нормализация данных с помощью статистики ImageNet"""
    imagenet_stats = np.array([[0.485, 0.456, 0.406], [0.229, 0.224, 0.225]])
    return (im - imagenet_stats[0]) / imagenet_stats[1]


class RoadDataset(Dataset):
    def __init__(self, paths, bb, y, is_transforms=False):
        self.is_transforms = is_transforms
        self.paths = paths.values
        self.bb = bb.values
        self.y = y.values
        
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        y_class = self.y[idx]
        x, y_bb = transformsXY(path, self.bb[idx], self.is_transforms)
        x = normalize(x)
        x = np.rollaxis(x, 2)
        return x, y_class, y_bb

    
train_ds = RoadDataset(X_train['nom'], X_train['new_bb'], y_train, is_transforms=True)
valid_ds = RoadDataset(X_val['nom'], X_val['new_bb'], y_val)
print(train_ds)

                        nom                        new_bb
0     train/images/0495.jpg   [135.0, 59.0, 185.0, 125.0]
1     train/images/0481.jpg    [59.0, 68.0, 161.0, 188.0]
2     train/images/0318.jpg  [118.0, 114.0, 140.0, 190.0]
3     train/images/0318.jpg  [126.0, 257.0, 134.0, 289.0]
4     train/images/0324.jpg  [119.0, 139.0, 162.0, 189.0]
...                     ...                           ...
1129  train/images/0467.jpg  [126.0, 243.0, 142.0, 266.0]
1130  train/images/0467.jpg  [116.0, 205.0, 125.0, 217.0]
1131  train/images/0473.jpg  [105.0, 163.0, 128.0, 212.0]
1132  train/images/0315.jpg   [149.0, 64.0, 197.0, 110.0]
1133  train/images/0498.jpg  [125.0, 161.0, 156.0, 201.0]

[1134 rows x 2 columns]                         nom                        new_bb
12    train/images/0697.jpg  [182.0, 177.0, 189.0, 190.0]
381   train/images/0807.jpg      [57.0, 30.0, 73.0, 47.0]
717   train/images/0749.jpg  [257.0, 103.0, 272.0, 113.0]
497   train/images/0037.jpg  [103.0, 257.0, 107

In [22]:
batch_size = 16
train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=batch_size)

In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 32 * 32, 512)
        self.fc2 = nn.Linear(512, 8)  # 2 classes pour la détection binaire

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 28 * 28)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [31]:
model = CNN()
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=0.006)
epochs = 1
model

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=65536, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=8, bias=True)
)

In [27]:
model.train()

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=65536, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=8, bias=True)
)

In [28]:
model.eval()

CNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=65536, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=8, bias=True)
)

In [36]:
loss_fn = torch.nn.CrossEntropyLoss()

# NB: Loss functions expect data in batches, so we're creating batches of 4
# Represents the model's confidence in each of the 10 classes for a given input
dummy_outputs = torch.rand(4, 10)
# Represents the correct class among the 10 being tested
dummy_labels = torch.tensor([1, 5, 3, 7])

print(dummy_outputs)
print(dummy_labels)

loss = loss_fn(dummy_outputs, dummy_labels)
print('Total loss for this batch: {}'.format(loss.item()))

: 

In [ ]:
# Optimizers specified in the torch.optim package
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [ ]:
def train_one_epoch(epoch_index, tb_writer):
    running_loss = 0.
    last_loss = 0.

    # Here, we use enumerate(train_dl) instead of
    # iter(train_dl) so that we can track the batch
    # index and do some intra-epoch reporting
    for i, data in enumerate(train_dl):
        # Every data instance is an input + label pair
        inputs, labels = data

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model(inputs)

        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

        # Gather data and report
        running_loss += loss.item()
        if i % 1000 == 999:
            last_loss = running_loss / 1000 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            tb_x = epoch_index * len(train_dl) + i + 1
            tb_writer.add_scalar('Loss/train', last_loss, tb_x)
            running_loss = 0.

    return last_loss